# 01b — Kalibrasi Mesin (Pengujian Linear)

Bagian ini mengolah dataset **pengujian linear** menjadi nilai kalibrasi mesin.

**Alur:**
1. Muat `K` dan `D` hasil `01_camera_calibration_paper/kalibrasi_kamera.ipynb`
2. Baca triplet citra `oriy{i}.png` / `fwd{i}.png` / `bck{i}.png` (i = 1..40)
3. `solvePnP` per citra -> `tvec` (posisi mm), disaring dengan ambang RMSE
4. Hitung rata-rata dX/dY, error terhadap target, backlash, dan skew
5. Cetak rekomendasi firmware: `M92` (steps/mm), `M425` (backlash), `M852` (skew)

**Cara pakai baseline vs hasil kalibrasi:** jalankan dulu dengan `CURRENT_STEPS_MM`
dan folder baseline, lalu ganti `base_path` + `CURRENT_STEPS_MM` ke folder hasil
kalibrasi untuk melihat sisa error setelah kalibrasi.

> Sel 10 (`CARI NOISE`) dan sel 11 (`cek_frame_folder`) adalah utilitas bantu,
> bukan bagian dari perhitungan kalibrasi.


In [ ]:
# ==========================================================================
# 1. MUAT KONFIGURASI PATH  (satu sumber: config_paths.py di akar repo)
# ==========================================================================
import os
import sys

_p = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_p, "config_paths.py")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError(
            "config_paths.py tidak ditemukan. Jalankan notebook ini dari dalam folder "
            "repo hasil penataan, atau set environment KALIBRASI_REPO ke akar repo."
        )
    _p = _parent
sys.path.insert(0, _p)
import config_paths as CFG

from calibration_pipeline import (
    detect_edlines_raw, get_initial_corners_strict_paper,
    optimize_subpixel_gradient, sort_corners_robust_paper,
)

CFG.diagnose()
print()

# ==========================================================================
# 2. PARAMETER MESIN PER MODE PENGUJIAN
# ==========================================================================
# Ganti mode tanpa mengedit notebook:
#     Windows : set KALIBRASI_MODE=baseline
#     Linux   : export KALIBRASI_MODE=baseline
#   baseline          -> sebelum kalibrasi, steps/mm 80,          target 10 mm
#   calibrated        -> setelah kalibrasi, steps/mm 79.72/80.40,  target 10 mm
#   measurement_error -> dataset 0.5 mm, baseline semua, pembanding dial gauge
RUN_MODE = CFG.RUN_MODE
P = CFG.MODE_PARAMS[RUN_MODE]
print(f"MODE: {RUN_MODE}  ->  delta_x={P['delta_x']}  delta_y={P['delta_y']}  "
      f"steps_x={P['steps_x']}  steps_y={P['steps_y']}")
print(f"  dataset X : {CFG.DATASET_X}")
print(f"  dataset Y : {CFG.DATASET_Y}")
print()

# ==========================================================================
# 3. MUAT PARAMETER KAMERA (K dan D) DARI CSV HASIL KALIBRASI KAMERA
# ==========================================================================
# Dibaca dari CSV keluaran 01_camera_calibration_paper/kalibrasi_kamera.ipynb,
# jadi kalibrasi kamera tidak perlu dijalankan ulang.
import numpy as np
import pandas as pd

CAM_DIR = os.path.join(CFG.OUTPUT_DIR, "camera_calibration")
_HINT = ("Jalankan dulu 01_camera_calibration_paper/kalibrasi_kamera.ipynb sampai "
         "sel ekspor selesai supaya CSV parameter kamera terbentuk.")

if "camera_matrix" in globals() and "dist_coeffs" in globals():
    print("Parameter kamera sudah ada di memori kernel.")
else:
    for _f in ("camera_matrix.csv", "dist_coeffs.csv", "camera_params.csv"):
        if not os.path.isfile(os.path.join(CAM_DIR, _f)):
            raise FileNotFoundError(f"Tidak ditemukan: {os.path.join(CAM_DIR, _f)}\n{_HINT}")

    camera_matrix = np.loadtxt(os.path.join(CAM_DIR, "camera_matrix.csv"),
                               delimiter=",", ndmin=2)
    dist_coeffs = np.loadtxt(os.path.join(CAM_DIR, "dist_coeffs.csv"),
                             delimiter=",", ndmin=2).reshape(1, -1)

    _p = pd.read_csv(os.path.join(CAM_DIR, "camera_params.csv")).set_index("parameter")["value"]
    img_shape = (int(_p["image_width"]), int(_p["image_height"]))

    print(f"Parameter kamera dimuat dari: {CAM_DIR}")
    print(f"  image_size  : {img_shape}")
    print("  K =\n", np.round(camera_matrix, 4))
    print("  dist_coeffs =", np.round(dist_coeffs.ravel(), 6))
print()


In [ ]:
# EKSTRAKSI POSISI, ANALISIS STATISTIK, DAN KALIBRASI FIRMWARE (STEPS/MM, BACKLASH, SKEW) X
# DENGAN FITUR VISUALISASI OTOMATIS PADA SAMPEL VALID PERTAMA
# =========================================================================================
import cv2
import numpy as np
import os

# --- 1. MEMUAT PARAMETER KAMERA ---
try:
    K = camera_matrix 
    D = dist_coeffs
except NameError:
    raise NameError("ERROR: Jalankan Cell Kalibrasi Kamera terlebih dahulu!")

CHESSBOARD_SIZE = CFG.CHESSBOARD_SIZE
SQUARE_SIZE = CFG.SQUARE_SIZE_MM  # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. KONFIGURASI INPUT & PARAMETER MESIN ---
base_path = CFG.DATASET_X   # diatur lewat config_paths.py

# Parameter Target & Mesin
EXPECTED_DELTA_X = P["delta_x"] if True else 0.0
EXPECTED_DELTA_Y = P["delta_y"] if False else 0.0
CURRENT_STEPS_MM = P["steps_x"]
RMSE_THRESHOLD = 0.2       # Batas maksimal RMSE yang diizinkan
MAX_VALID_SAMPLES = 10     # Batas jumlah sampel valid yang dibutuhkan

FILE_INDICES = list(range(1, 41))

# Membuat triplet gambar: (Origin, Forward, Backward/Return)
IMAGE_SETS = [
    (
        os.path.join(base_path, f"oriy{i}.png"),
        os.path.join(base_path, f"fwd{i}.png"),
        os.path.join(base_path, f"bck{i}.png")
    )
    for i in FILE_INDICES
]

# --- 3. FUNGSI EKSTRAKSI POSE (TERINTEGRASI FITUR DRAWING) ---
def get_pose_from_image(image_path, draw_debug=False):
    if not os.path.exists(image_path):
        return None, None, None
        
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Buat kanvas gambar: Grayscale tapi 3-channel agar warna merah bisa digambar
    img_draw = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if draw_debug else None
    
    # Pipeline EDLines (Asumsi fungsi-fungsi ini sudah ada di memori/cell sebelumnya)
    lines = detect_edlines_raw(gray)
    initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            # --- BLOK DRAWING SUSUNAN CORNER ---
            if draw_debug:
                for idx, pt in enumerate(sorted_corners):
                    # Menggambar titik sudut (merah)
                    cv2.circle(img_draw, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)
                    # Menambahkan teks indeks urutan sudut
                    cv2.putText(img_draw, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (0, 0, 255), 2)

            corners_final = sorted_corners.reshape(-1, 1, 2)
            success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
            
            if success:
                imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                return tvec, rmse, img_draw
                
    return None, None, img_draw

# --- 4. EKSEKUSI BATCH & ANALISIS ---
print("="*70)
print("ANALISIS POSISI, PERPINDAHAN, & REPROJECTION ERROR")
print("="*70)

# List penampung data statistik
list_delta_x = []
list_delta_y = []
list_error_x = []
list_error_y = []
list_rmse = []
list_backlash = []
list_skew = []

# List penampung data yang gagal
list_sampel_gagal = []
berhasil = 0

# Flag penanda apakah visualisasi sampel valid pertama sudah disimpan
visualisasi_tersimpan = False

for idx, (ori_path, fwd_path, bck_path) in enumerate(IMAGE_SETS):
    sampel_id = FILE_INDICES[idx]
    print(f"\n[SAMPEL {sampel_id}]")
    
    # Aktifkan mode visualisasi jika belum ada sampel valid yang berhasil disimpan
    is_visualize = not visualisasi_tersimpan
    
    tvec_ori, rmse_ori, img_ori_drawn = get_pose_from_image(ori_path, draw_debug=is_visualize)
    tvec_fwd, rmse_fwd, img_fwd_drawn = get_pose_from_image(fwd_path, draw_debug=is_visualize)
    tvec_bck, rmse_bck, img_bck_drawn = get_pose_from_image(bck_path, draw_debug=is_visualize)
    
    # --- LOGIKA VALIDASI ---
    # 1. Cek apakah ada gambar yang gagal diproses (None)
    if tvec_ori is None or tvec_fwd is None or tvec_bck is None:
        print("  [GAGAL] Salah satu atau lebih gambar gagal diproses (pose tidak ditemukan).")
        list_sampel_gagal.append(sampel_id)
        continue 
        
    # 2. Cek apakah ada RMSE yang melebihi batas
    if rmse_ori > RMSE_THRESHOLD or rmse_fwd > RMSE_THRESHOLD or rmse_bck > RMSE_THRESHOLD:
        print(f"  [GAGAL] Data ditolak karena RMSE > {RMSE_THRESHOLD}.")
        print(f"          Detail RMSE -> Ori: {rmse_ori:.4f}, Fwd: {rmse_fwd:.4f}, Bck: {rmse_bck:.4f}")
        list_sampel_gagal.append(sampel_id)
        continue 

    # --- JIKA LOLOS VALIDASI, EKSEKUSI DATA ---
    berhasil += 1
    
    x_orig, y_orig = tvec_ori[0][0], tvec_ori[1][0]
    x_fwd, y_fwd = tvec_fwd[0][0], tvec_fwd[1][0]
    x_bck, y_bck = tvec_bck[0][0], tvec_bck[1][0]
    
    # Kalkulasi Delta Maju
    delta_x = x_fwd - x_orig
    delta_y = y_fwd - y_orig
    
    # Kalkulasi Error dari Expected
    error_x = abs(delta_x - EXPECTED_DELTA_X)
    error_y = abs(delta_y - EXPECTED_DELTA_Y)
    
    # Kalkulasi Backlash
    backlash_val = abs(x_bck - x_orig)
    
    # Kalkulasi Skew Factor (Y drift / X move)
    skew_factor = delta_y / delta_x if delta_x != 0 else 0
    
    # Simpan metrik dasar ke list
    list_delta_x.append(delta_x)
    list_delta_y.append(delta_y)
    list_error_x.append(error_x)
    list_error_y.append(error_y)
    list_skew.append(skew_factor)
    list_backlash.append(backlash_val)
    list_rmse.extend([rmse_ori, rmse_fwd, rmse_bck])
    
    # Output ke layar
    print(f"  > Posisi Awal  : X = {x_orig:8.4f} mm, Y = {y_orig:8.4f} mm | RMSE = {rmse_ori:.4f} px")
    print(f"  > Posisi Maju  : X = {x_fwd:8.4f} mm, Y = {y_fwd:8.4f} mm | RMSE = {rmse_fwd:.4f} px")
    print(f"  > Posisi Balik : X = {x_bck:8.4f} mm, Y = {y_bck:8.4f} mm | RMSE = {rmse_bck:.4f} px")
    print(f"  > PERPINDAHAN  : dX = {delta_x:8.4f} mm (Err: {error_x:.4f}), dY = {delta_y:8.4f} mm (Err: {error_y:.4f})")
    print(f"  > BACKLASH (X) : {backlash_val:8.4f} mm")

    # # --- BLOK VISUALISASI & SIMPAN PNG (KHUSUS SAMPEL VALID PERTAMA) INI FONTNYA KEKECILAN ---
    # if is_visualize and img_ori_drawn is not None and img_fwd_drawn is not None and img_bck_drawn is not None:
    #     font = cv2.FONT_HERSHEY_SIMPLEX
        
    #     # Overlay Teks - Frame Origin
    #     cv2.putText(img_ori_drawn, "Origin Frame", (30, 50), font, 1.2, (0, 0, 255), 3)
    #     cv2.putText(img_ori_drawn, f"X: {x_orig:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_ori_drawn, f"Y: {y_orig:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_ori_drawn, f"RMSE: {rmse_ori:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
    #     # Overlay Teks - Frame Forward
    #     cv2.putText(img_fwd_drawn, "Forward Frame", (30, 50), font, 1.2, (0, 0, 255), 3)
    #     cv2.putText(img_fwd_drawn, f"X: {x_fwd:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_fwd_drawn, f"Y: {y_fwd:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_fwd_drawn, f"RMSE: {rmse_fwd:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
    #     # Overlay Teks - Frame Backward
    #     cv2.putText(img_bck_drawn, "Backward Frame", (30, 50), font, 1.2, (0, 0, 255), 3)
    #     cv2.putText(img_bck_drawn, f"X: {x_bck:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_bck_drawn, f"Y: {y_bck:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_bck_drawn, f"RMSE: {rmse_bck:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
    #     # Simpan ketiga file gambar
    #     cv2.imwrite("visualisasi_ori.png", img_ori_drawn)
    #     cv2.imwrite("visualisasi_fwd.png", img_fwd_drawn)
    #     cv2.imwrite("visualisasi_bck.png", img_bck_drawn)
        
    #     print("\n  [INFO] Gambar 'visualisasi_ori.png', 'visualisasi_fwd.png', dan 'visualisasi_bck.png' berhasil disimpan!")

    # --- BLOK VISUALISASI & SIMPAN PNG (KHUSUS SAMPEL VALID PERTAMA) ---
    if is_visualize and img_ori_drawn is not None and img_fwd_drawn is not None and img_bck_drawn is not None:
        font = cv2.FONT_HERSHEY_SIMPLEX
        
        # Pengaturan Skala dan Ketebalan Font (Diperbesar)
        scale_title = 1.8   # Semakin besar nilainya, semakin besar judulnya
        scale_text = 1.3    # Semakin besar nilainya, semakin besar teks datanya
        thick_title = 4
        thick_text = 3

        # Overlay Teks - Frame Origin
        cv2.putText(img_ori_drawn, "Origin Frame", (30, 60), font, scale_title, (0, 0, 255), thick_title)
        cv2.putText(img_ori_drawn, f"X: {x_orig:.3f} mm", (30, 120), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_ori_drawn, f"Y: {y_orig:.3f} mm", (30, 180), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_ori_drawn, f"RMSE: {rmse_ori:.4f} px", (30, 240), font, scale_text, (255, 255, 0), thick_text)
        
        # Overlay Teks - Frame Forward
        cv2.putText(img_fwd_drawn, "Forward Frame", (30, 60), font, scale_title, (0, 0, 255), thick_title)
        cv2.putText(img_fwd_drawn, f"X: {x_fwd:.3f} mm", (30, 120), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_fwd_drawn, f"Y: {y_fwd:.3f} mm", (30, 180), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_fwd_drawn, f"RMSE: {rmse_fwd:.4f} px", (30, 240), font, scale_text, (255, 255, 0), thick_text)
        
        # Overlay Teks - Frame Backward
        cv2.putText(img_bck_drawn, "Backward Frame", (30, 60), font, scale_title, (0, 0, 255), thick_title)
        cv2.putText(img_bck_drawn, f"X: {x_bck:.3f} mm", (30, 120), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_bck_drawn, f"Y: {y_bck:.3f} mm", (30, 180), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_bck_drawn, f"RMSE: {rmse_bck:.4f} px", (30, 240), font, scale_text, (255, 255, 0), thick_text)
        
        # Simpan ketiga file gambar
        cv2.imwrite("visualisasi_ori.png", img_ori_drawn)
        cv2.imwrite("visualisasi_fwd.png", img_fwd_drawn)
        cv2.imwrite("visualisasi_bck.png", img_bck_drawn)
        
        print("\n  [INFO] Gambar 'visualisasi_ori.png', 'visualisasi_fwd.png', dan 'visualisasi_bck.png' berhasil disimpan!")
        
        # Tandai bahwa visualisasi sudah berhasil diambil untuk mencegah draw_debug di iterasi selanjutnya
        visualisasi_tersimpan = True

    # --- CEK BATAS MAKSIMAL SAMPEL ---
    if berhasil >= MAX_VALID_SAMPLES:
        print(f"\n[INFO] Telah mencapai target {MAX_VALID_SAMPLES} sampel valid. Proses ekstraksi dihentikan.")
        break

# --- 5. RINGKASAN STATISTIK & REKOMENDASI FIRMWARE ---
print("\n" + "="*70)
print("HASIL AKHIR STATISTIK & REKOMENDASI KALIBRASI FIRMWARE")
print("="*70)

if berhasil > 0:
    # Perhitungan Statistik Rata-rata & Standar Deviasi
    avg_dx, std_dx = np.mean(list_delta_x), np.std(list_delta_x)
    avg_dy, std_dy = np.mean(list_delta_y), np.std(list_delta_y)
    
    avg_err_x, std_err_x = np.mean(list_error_x), np.std(list_error_x)
    avg_err_y, std_err_y = np.mean(list_error_y), np.std(list_error_y)
    
    avg_rmse, std_rmse = np.mean(list_rmse), np.std(list_rmse)
    
    print(f"Jumlah Sampel Valid/Berhasil : {berhasil} dari total {len(IMAGE_SETS)} yang tersedia\n")
    
    print("--- 1. STATISTIK PERPINDAHAN & ERROR ---")
    print(f"Target Expected      : X = {EXPECTED_DELTA_X:.4f} mm, Y = {EXPECTED_DELTA_Y:.4f} mm")
    print(f"Rata-rata Delta X    : {avg_dx:8.4f} mm (Std Dev: {std_dx:.4f} mm)")
    print(f"Rata-rata Delta Y    : {avg_dy:8.4f} mm (Std Dev: {std_dy:.4f} mm)")
    print(f"Rata-rata Error X    : {avg_err_x:8.4f} mm (Std Dev: {std_err_x:.4f} mm)")
    print(f"Rata-rata Error Y    : {avg_err_y:8.4f} mm (Std Dev: {std_err_y:.4f} mm)")
    print(f"Rata-rata RMSE       : {avg_rmse:8.4f} px (Std Dev: {std_rmse:.4f} px)")
    
    print("\n--- 2. REKOMENDASI UPDATE STEPS/MM ---")
    if EXPECTED_DELTA_X != 0:
        new_steps_x = (EXPECTED_DELTA_X / avg_dx) * CURRENT_STEPS_MM
        print(f"Current Steps/mm (X) : {CURRENT_STEPS_MM:.4f}")
        print(f"-> NEW Steps/mm (X)  : {new_steps_x:.4f}  (G-Code: M92 X{new_steps_x:.2f})")
    else:
        print("-> Update Steps/mm X : Dilewati (Target Delta X = 0)")

    print("\n--- 3. REKOMENDASI BACKLASH COMPENSATION ---")
    avg_backlash = np.mean(list_backlash)
    std_backlash = np.std(list_backlash)
    print(f"Rata-rata Backlash X : {avg_backlash:8.4f} mm (Std Dev: {std_backlash:.4f} mm)")
    print(f"-> Firmware Update   : G-Code: M425 X{avg_backlash:.4f}")

    print("\n--- 4. REKOMENDASI SKEW COMPENSATION (XY ORTHOGONALITY) ---")
    avg_skew = np.mean(list_skew)
    angle_deg = np.degrees(np.arctan(avg_skew))
    print(f"Rata-rata Skew (I)   : {avg_skew:.6f}")
    print(f"Kemiringan Sumbu     : {angle_deg:.4f} Derajat")
    if abs(avg_skew) < 0.001:
        print("-> Status Skew       : Sangat Siku (Orthogonal). Kompensasi tidak diperlukan.")
    else:
        print(f"-> Firmware Update   : G-Code: M852 I{avg_skew:.6f}")

else:
    print("Tidak ada data valid yang lolos seleksi (RMSE > 0.2 atau gagal proses).")

# --- MENAMPILKAN SAMPEL YANG GAGAL ---
print("\n--- 5. STATUS SAMPEL GAGAL ---")
if len(list_sampel_gagal) > 0:
    print(f"Terdapat {len(list_sampel_gagal)} sampel yang ditolak/gagal sebelum mencapai {MAX_VALID_SAMPLES} data valid:")
    print(f"-> {list_sampel_gagal}")
else:
    print(f"-> Tidak ada sampel yang gagal sebelum mencapai target {MAX_VALID_SAMPLES} data valid!")
print("="*70)

In [ ]:
# EKSTRAKSI POSISI, ANALISIS STATISTIK, & KALIBRASI SUMBU Y
# DENGAN FITUR VISUALISASI OTOMATIS PADA SAMPEL VALID PERTAMA
# =========================================================
import cv2
import numpy as np
import os

# --- 1. MEMUAT PARAMETER KAMERA ---
try:
    K = camera_matrix 
    D = dist_coeffs
except NameError:
    raise NameError("ERROR: Jalankan Cell Kalibrasi Kamera terlebih dahulu!")

CHESSBOARD_SIZE = CFG.CHESSBOARD_SIZE
SQUARE_SIZE = CFG.SQUARE_SIZE_MM  # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. KONFIGURASI INPUT & PARAMETER MESIN ---
# Sesuaikan base_path dengan folder dataset Sumbu Y
base_path = CFG.DATASET_Y   # diatur lewat config_paths.py

# Parameter Target & Mesin (Fokus Sumbu Y)
EXPECTED_DELTA_X = P["delta_x"] if False else 0.0
EXPECTED_DELTA_Y = P["delta_y"] if True else 0.0
CURRENT_STEPS_MM = P["steps_y"]
RMSE_THRESHOLD = 0.2       # Batas maksimal RMSE yang diizinkan
MAX_VALID_SAMPLES = 10     # Batas jumlah sampel valid yang dibutuhkan

FILE_INDICES = list(range(1, 41))

# Membuat triplet gambar untuk Sumbu Y: (Origin, Forward, Backward/Return)
IMAGE_SETS = [
    (
        os.path.join(base_path, f"oriy{i}.png"),
        os.path.join(base_path, f"fwd{i}.png"),
        os.path.join(base_path, f"bck{i}.png")
    )
    for i in FILE_INDICES
]

# --- 3. FUNGSI EKSTRAKSI POSE (TERINTEGRASI FITUR DRAWING) ---
def get_pose_from_image(image_path, draw_debug=False):
    if not os.path.exists(image_path):
        return None, None, None
        
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Buat kanvas gambar: Grayscale tapi 3-channel agar warna merah bisa digambar
    img_draw = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if draw_debug else None
    
    # Pipeline EDLines (Asumsi fungsi-fungsi ini sudah ada di memori/cell sebelumnya)
    lines = detect_edlines_raw(gray)
    initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            # --- BLOK DRAWING SUSUNAN CORNER ---
            if draw_debug:
                for idx, pt in enumerate(sorted_corners):
                    # Menggambar titik sudut (merah)
                    cv2.circle(img_draw, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)
                    # Menambahkan teks indeks urutan sudut
                    cv2.putText(img_draw, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (0, 0, 255), 2)

            corners_final = sorted_corners.reshape(-1, 1, 2)
            success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
            
            if success:
                imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                return tvec, rmse, img_draw
                
    return None, None, img_draw

# --- 4. EKSEKUSI BATCH & ANALISIS ---
print("="*70)
print("ANALISIS POSISI, PERPINDAHAN, & REPROJECTION ERROR (SUMBU Y)")
print("="*70)

# List penampung data statistik
list_delta_x = []
list_delta_y = []
list_error_x = []
list_error_y = []
list_rmse = []
list_backlash = []
list_skew = []

# List penampung data yang gagal
list_sampel_gagal = []
berhasil = 0

# Flag penanda apakah visualisasi sampel valid pertama sudah disimpan
visualisasi_tersimpan = False

for idx, (ori_path, fwd_path, bck_path) in enumerate(IMAGE_SETS):
    sampel_id = FILE_INDICES[idx]
    print(f"\n[SAMPEL {sampel_id}]")
    
    # Aktifkan mode visualisasi jika belum ada sampel valid yang berhasil disimpan
    is_visualize = not visualisasi_tersimpan
    
    tvec_ori, rmse_ori, img_ori_drawn = get_pose_from_image(ori_path, draw_debug=is_visualize)
    tvec_fwd, rmse_fwd, img_fwd_drawn = get_pose_from_image(fwd_path, draw_debug=is_visualize)
    tvec_bck, rmse_bck, img_bck_drawn = get_pose_from_image(bck_path, draw_debug=is_visualize)
    
    # --- LOGIKA VALIDASI ---
    # 1. Cek apakah ada gambar yang gagal diproses (None)
    if tvec_ori is None or tvec_fwd is None or tvec_bck is None:
        print("  [GAGAL] Salah satu atau lebih gambar gagal diproses (pose tidak ditemukan).")
        list_sampel_gagal.append(sampel_id)
        continue  # Langsung lanjut ke iterasi sampel berikutnya
        
    # 2. Cek apakah ada RMSE yang melebihi batas
    if rmse_ori > RMSE_THRESHOLD or rmse_fwd > RMSE_THRESHOLD or rmse_bck > RMSE_THRESHOLD:
        print(f"  [GAGAL] Data ditolak karena RMSE > {RMSE_THRESHOLD}.")
        print(f"          Detail RMSE -> Ori: {rmse_ori:.4f}, Fwd: {rmse_fwd:.4f}, Bck: {rmse_bck:.4f}")
        list_sampel_gagal.append(sampel_id)
        continue  # Langsung lanjut ke iterasi sampel berikutnya

    # --- JIKA LOLOS VALIDASI, EKSEKUSI DATA ---
    berhasil += 1
    
    x_orig, y_orig = tvec_ori[0][0], tvec_ori[1][0]
    x_fwd, y_fwd = tvec_fwd[0][0], tvec_fwd[1][0]
    x_bck, y_bck = tvec_bck[0][0], tvec_bck[1][0]
    
    # Kalkulasi Delta Maju
    delta_x = x_fwd - x_orig
    delta_y = y_fwd - y_orig
    
    # Kalkulasi Error dari Expected
    error_x = abs(delta_x - EXPECTED_DELTA_X)
    error_y = abs(delta_y - EXPECTED_DELTA_Y)
    
    # Kalkulasi Backlash (Fokus Sumbu Y: selisih posisi Y akhir dengan Y awal)
    backlash_val = abs(y_bck - y_orig)
    
    # Kalkulasi Skew Factor (X drift / Y move)
    # Jika bergerak di sumbu Y, skew adalah seberapa jauh sumbu X bergeser
    skew_factor = delta_x / delta_y if delta_y != 0 else 0
    
    # Simpan metrik dasar ke list
    list_delta_x.append(delta_x)
    list_delta_y.append(delta_y)
    list_error_x.append(error_x)
    list_error_y.append(error_y)
    list_skew.append(skew_factor)
    list_backlash.append(backlash_val)
    list_rmse.extend([rmse_ori, rmse_fwd, rmse_bck])
    
    # Output ke layar
    print(f"  > Posisi Awal  : X = {x_orig:8.4f} mm, Y = {y_orig:8.4f} mm | RMSE = {rmse_ori:.4f} px")
    print(f"  > Posisi Maju  : X = {x_fwd:8.4f} mm, Y = {y_fwd:8.4f} mm | RMSE = {rmse_fwd:.4f} px")
    print(f"  > Posisi Balik : X = {x_bck:8.4f} mm, Y = {y_bck:8.4f} mm | RMSE = {rmse_bck:.4f} px")
    print(f"  > PERPINDAHAN  : dX = {delta_x:8.4f} mm (Err: {error_x:.4f}), dY = {delta_y:8.4f} mm (Err: {error_y:.4f})")
    print(f"  > BACKLASH (Y) : {backlash_val:8.4f} mm")

    # --- BLOK VISUALISASI & SIMPAN PNG (KHUSUS SAMPEL VALID PERTAMA) ---
    if is_visualize and img_ori_drawn is not None and img_fwd_drawn is not None and img_bck_drawn is not None:
        font = cv2.FONT_HERSHEY_SIMPLEX
        
        # Overlay Teks - Frame Origin
        cv2.putText(img_ori_drawn, "FRAME AWAL (ORIGIN)", (30, 50), font, 1.2, (0, 0, 255), 3)
        cv2.putText(img_ori_drawn, f"X: {x_orig:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_ori_drawn, f"Y: {y_orig:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_ori_drawn, f"RMSE: {rmse_ori:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
        # Overlay Teks - Frame Forward
        cv2.putText(img_fwd_drawn, "FRAME MAJU (FORWARD)", (30, 50), font, 1.2, (0, 0, 255), 3)
        cv2.putText(img_fwd_drawn, f"X: {x_fwd:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_fwd_drawn, f"Y: {y_fwd:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_fwd_drawn, f"RMSE: {rmse_fwd:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
        # Overlay Teks - Frame Backward
        cv2.putText(img_bck_drawn, "FRAME BALIK (BACKWARD)", (30, 50), font, 1.2, (0, 0, 255), 3)
        cv2.putText(img_bck_drawn, f"X: {x_bck:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_bck_drawn, f"Y: {y_bck:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_bck_drawn, f"RMSE: {rmse_bck:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
        # Simpan ketiga file gambar (Diberi tag _Y_ agar tidak menimpa Sumbu X)
        cv2.imwrite("visualisasi_Y_ori.png", img_ori_drawn)
        cv2.imwrite("visualisasi_Y_fwd.png", img_fwd_drawn)
        cv2.imwrite("visualisasi_Y_bck.png", img_bck_drawn)
        
        print("\n  [INFO] Gambar 'visualisasi_Y_ori.png', 'visualisasi_Y_fwd.png', dan 'visualisasi_Y_bck.png' berhasil disimpan!")
        
        # Tandai bahwa visualisasi sudah berhasil diambil untuk mencegah draw_debug di iterasi selanjutnya
        visualisasi_tersimpan = True

    # --- CEK BATAS MAKSIMAL SAMPEL ---
    if berhasil >= MAX_VALID_SAMPLES:
        print(f"\n[INFO] Telah mencapai target {MAX_VALID_SAMPLES} sampel valid. Proses ekstraksi dihentikan.")
        break

# --- 5. RINGKASAN STATISTIK & REKOMENDASI FIRMWARE ---
print("\n" + "="*70)
print("HASIL AKHIR STATISTIK & REKOMENDASI KALIBRASI FIRMWARE")
print("="*70)

if berhasil > 0:
    # Perhitungan Statistik Rata-rata & Standar Deviasi
    avg_dx, std_dx = np.mean(list_delta_x), np.std(list_delta_x)
    avg_dy, std_dy = np.mean(list_delta_y), np.std(list_delta_y)
    
    avg_err_x, std_err_x = np.mean(list_error_x), np.std(list_error_x)
    avg_err_y, std_err_y = np.mean(list_error_y), np.std(list_error_y)
    
    avg_rmse, std_rmse = np.mean(list_rmse), np.std(list_rmse)
    
    print(f"Jumlah Sampel Valid/Berhasil : {berhasil} dari total {len(IMAGE_SETS)} yang tersedia\n")
    
    print("--- 1. STATISTIK PERPINDAHAN & ERROR ---")
    print(f"Target Expected      : X = {EXPECTED_DELTA_X:.4f} mm, Y = {EXPECTED_DELTA_Y:.4f} mm")
    print(f"Rata-rata Delta X    : {avg_dx:8.4f} mm (Std Dev: {std_dx:.4f} mm)")
    print(f"Rata-rata Delta Y    : {avg_dy:8.4f} mm (Std Dev: {std_dy:.4f} mm)")
    print(f"Rata-rata Error X    : {avg_err_x:8.4f} mm (Std Dev: {std_err_x:.4f} mm)")
    print(f"Rata-rata Error Y    : {avg_err_y:8.4f} mm (Std Dev: {std_err_y:.4f} mm)")
    print(f"Rata-rata RMSE       : {avg_rmse:8.4f} px (Std Dev: {std_rmse:.4f} px)")
    
    print("\n--- 2. REKOMENDASI UPDATE STEPS/MM ---")
    if EXPECTED_DELTA_Y != 0:
        new_steps_y = (EXPECTED_DELTA_Y / avg_dy) * CURRENT_STEPS_MM
        print(f"Current Steps/mm (Y) : {CURRENT_STEPS_MM:.4f}")
        print(f"-> NEW Steps/mm (Y)  : {new_steps_y:.4f}  (G-Code: M92 Y{new_steps_y:.2f})")
    else:
        print("-> Update Steps/mm Y : Dilewati (Target Delta Y = 0)")

    print("\n--- 3. REKOMENDASI BACKLASH COMPENSATION ---")
    avg_backlash = np.mean(list_backlash)
    std_backlash = np.std(list_backlash)
    print(f"Rata-rata Backlash Y : {avg_backlash:8.4f} mm (Std Dev: {std_backlash:.4f} mm)")
    print(f"-> Firmware Update   : G-Code: M425 Y{avg_backlash:.4f}")

    print("\n--- 4. REKOMENDASI SKEW COMPENSATION (XY ORTHOGONALITY) ---")
    avg_skew = np.mean(list_skew)
    angle_deg = np.degrees(np.arctan(avg_skew))
    print(f"Rata-rata Skew (I)   : {avg_skew:.6f}")
    print(f"Kemiringan Sumbu     : {angle_deg:.4f} Derajat")
    if abs(avg_skew) < 0.001:
        print("-> Status Skew       : Sangat Siku (Orthogonal). Kompensasi tidak diperlukan.")
    else:
        print(f"-> Firmware Update   : G-Code: M852 I{avg_skew:.6f}")

else:
    print("Tidak ada data valid yang lolos seleksi (RMSE > 0.2 atau gagal proses).")

# --- MENAMPILKAN SAMPEL YANG GAGAL ---
print("\n--- 5. STATUS SAMPEL GAGAL ---")
if len(list_sampel_gagal) > 0:
    print(f"Terdapat {len(list_sampel_gagal)} sampel yang ditolak/gagal sebelum mencapai {MAX_VALID_SAMPLES} data valid:")
    print(f"-> {list_sampel_gagal}")
else:
    print(f"-> Tidak ada sampel yang gagal sebelum mencapai target {MAX_VALID_SAMPLES} data valid!")
print("="*70)

In [ ]:
# ==========================================================================
# [OPSIONAL / HISTORIS] CARI NOISE
# ==========================================================================
# Sel ini dulu dipakai untuk membandingkan satu citra acuan (calib_0000.png) dengan
# beberapa citra lain dari dataset "pengujian_noise_fix_banget" untuk mencari sumber
# derau. Dataset tersebut TIDAK ikut ke repo ini, sehingga base_path di bawah sengaja
# diarahkan ke lokasi yang belum ada: semua pasangan akan dilaporkan [GAGAL] dan
# tidak ada angka yang dihasilkan. Ini BUKAN error.
# Kalau ingin memakainya, arahkan base_path ke folder Anda sendiri yang berisi
# calib_0000.png, calib_0011.png, dst.
# ==========================================================================
#CARI NOISE
# =========================================================================================
import cv2
import numpy as np
import os

# --- 1. MEMUAT PARAMETER KAMERA ---
try:
    K = camera_matrix 
    D = dist_coeffs
except NameError:
    raise NameError("ERROR: Jalankan Cell Kalibrasi Kamera terlebih dahulu!")

CHESSBOARD_SIZE = (7, 7)
SQUARE_SIZE = 20.0 # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. KONFIGURASI INPUT GAMBAR ---
base_path = CFG.resolve("noise/pengujian_noise_fix_banget")   # belum ada di repo

IMAGE_PAIRS = [
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0011.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0019.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0013.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0004.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0018.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0006.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0007.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0008.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0019.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0010.png"))
]

# --- 3. FUNGSI EKSTRAKSI POSE (DENGAN GAMBAR KHUSUS SORTING) ---
def get_pose_from_image(image_path, draw_debug=False):
    if not os.path.exists(image_path):
        return None, None, None
        
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Buat kanvas gambar: Grayscale tapi 3-channel agar warna merah dan hijau bisa digambar
    img_draw = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if draw_debug else None
    
    lines = detect_edlines_raw(gray)
    initial_corners, valid_lines = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            # --- BLOK DRAWING: Persis seperti `img_f` kamu ---
            if draw_debug:
                for idx, pt in enumerate(sorted_corners):
                    # Ingat: cv2.imshow pakai BGR, jadi merah adalah (0, 0, 255)
                    cv2.circle(img_draw, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)
                    # Menggunakan font tipe 1 (FONT_HERSHEY_PLAIN) persis seperti kodemu sebelumnya
                    cv2.putText(img_draw, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (0, 0, 255), 2)

            corners_final = sorted_corners.reshape(-1, 1, 2)
            success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
            
            if success:
                imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                return tvec, rmse, img_draw
                
    return None, None, img_draw

# --- 4. EKSEKUSI BATCH & VISUALISASI ---
print("="*60)
print("ANALISIS POSISI & PERPINDAHAN")
print("="*60)

list_delta_x = []
list_delta_y = []

for idx, (origin_path, moved_path) in enumerate(IMAGE_PAIRS):
    print(f"\n[PASANGAN {idx+1}]")
    
    # Aktifkan visualisasi HANYA untuk pasangan gambar pertama
    is_visualize = (idx == 0) 
    
    tvec_origin, rmse_origin, img_orig_drawn = get_pose_from_image(origin_path, draw_debug=is_visualize)
    tvec_moved, rmse_moved, img_moved_drawn = get_pose_from_image(moved_path, draw_debug=is_visualize)
    
    if tvec_origin is not None and tvec_moved is not None:
        x_orig, y_orig = abs(tvec_origin[0][0]), abs(tvec_origin[1][0])
        x_moved, y_moved = abs(tvec_moved[0][0]), abs(tvec_moved[1][0])
        
        delta_x = x_moved - x_orig
        delta_y = y_moved - y_orig
        
        list_delta_x.append(delta_x)
        list_delta_y.append(delta_y)
        
        print(f"  > Gambar 1 (Awal)  : |X| = {x_orig:8.4f} mm, |Y| = {y_orig:8.4f} mm | RMSE = {rmse_origin:.4f} px")
        print(f"  > Gambar 2 (Akhir) : |X| = {x_moved:8.4f} mm, |Y| = {y_moved:8.4f} mm | RMSE = {rmse_moved:.4f} px")
        print(f"  > PERPINDAHAN      : Delta X = {delta_x:8.4f} mm, Delta Y = {delta_y:8.4f} mm")

        # --- BLOK PENAMBAHAN TEKS FRAME & KOORDINAT ---
        if is_visualize and img_orig_drawn is not None and img_moved_drawn is not None:
            font = cv2.FONT_HERSHEY_SIMPLEX
            
            # Teks Frame Awal
            cv2.putText(img_orig_drawn, "FRAME AWAL", (30, 50), font, 1.2, (0, 0, 255), 3) 
            cv2.putText(img_orig_drawn, f"X: {x_orig:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2) 
            cv2.putText(img_orig_drawn, f"Y: {y_orig:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2) 
            
            # Teks Frame Akhir
            cv2.putText(img_moved_drawn, "FRAME AKHIR", (30, 50), font, 1.2, (0, 0, 255), 3)
            cv2.putText(img_moved_drawn, f"X: {x_moved:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
            cv2.putText(img_moved_drawn, f"Y: {y_moved:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
            
            # Simpan gambar sebagai PNG terpisah
            cv2.imwrite("visualisasi_awal.png", img_orig_drawn)
            cv2.imwrite("visualisasi_akhir.png", img_moved_drawn)
            print("\n  [INFO] Gambar 'visualisasi_awal.png' dan 'visualisasi_akhir.png' berhasil disimpan!")
            
    else:
        print("  [GAGAL] Tidak dapat mengekstrak sudut dari salah satu atau kedua gambar.")

# --- 5. RINGKASAN STATISTIK ---
print("\n" + "="*60)
print("HASIL AKHIR ANALISIS RATA-RATA (DARI KOORDINAT ABSOLUT)")
print("="*60)

if len(list_delta_x) > 0:
    avg_dx = np.mean(list_delta_x)
    avg_dy = np.mean(list_delta_y)
    std_dx = np.std(list_delta_x)
    std_dy = np.std(list_delta_y)
    
    print(f"Jumlah Pasangan Berhasil : {len(list_delta_x)} / {len(IMAGE_PAIRS)}")
    print(f"Rata-rata Delta X        : {avg_dx:8.4f} mm (Std Dev: {std_dx:.4f})")
    print(f"Rata-rata Delta Y        : {avg_dy:8.4f} mm (Std Dev: {std_dy:.4f})")
else:
    print("Tidak ada data yang berhasil diproses untuk menghitung rata-rata.")

print("="*60)
print("SELESAI")

In [ ]:
import glob
import os
import cv2


def cek_frame_folder(folder_path):
    # Memastikan folder ada
    if not os.path.exists(folder_path):
        print(f"[!] Folder tidak ditemukan: {folder_path}")
        return

    # Cari semua file video umum (.avi, .mp4, .mov)
    extensions = ["*.avi", "*.mp4", "*.mov"]
    video_files = []
    for ext in extensions:
        video_files.extend(glob.glob(os.path.join(folder_path, ext)))

    # Urutkan nama file secara alfabetis
    video_files.sort()

    if not video_files:
        print(f"Tidak ada file video ditemukan di: {folder_path}")
        return

    print(f"\nMemindai folder: {folder_path}")
    print("=" * 65)
    print(f"{'Nama File':<30} | {'Resolusi':<15} | {'FPS':<6} | {'Total Frame'}")
    print("=" * 65)

    total_semua_frame = 0

    for vid_path in video_files:
        nama_file = os.path.basename(vid_path)
        cap = cv2.VideoCapture(vid_path)

        if not cap.isOpened():
            print(f"{nama_file:<30} | [!] Gagal membuka video")
            continue

        # Ambil metadata video
        lebar = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        tinggi = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        total_semua_frame += total_frame

        # Potong nama file jika terlalu panjang agar tabel tetap rapi
        nama_file_disp = (
            (nama_file[:27] + "...") if len(nama_file) > 30 else nama_file
        )

        print(
            f"{nama_file_disp:<30} | {lebar}x{tinggi:<10} | {fps:<6.0f} | {total_frame:,} frame"
        )

        cap.release()

    print("=" * 65)
    print(f"Total Keseluruhan Frame: {total_semua_frame:,} frame")
    print(f"Total Video Diproses   : {len(video_files)} video\n")


if __name__ == "__main__":
    # --- CARA PENGGUNAAN ---
    # Ganti path di bawah ini sesuai dengan nama folder video Anda.
    # Contoh: "dataset_otomatis_sesi_20260512_143000/video"
    TARGET_FOLDER = CFG.VIDEO_DIR  # diatur lewat config_paths.py

    cek_frame_folder(TARGET_FOLDER)